# Metadata formatting

earthkit-plots supports **metadata-aware format strings** in titles, legend labels, and axis labels. Curly-brace placeholders like `{variable_name}` or `{time:%B %Y}` are automatically populated from the data's metadata - regardless of whether it comes from GRIB, netCDF/xarray, or a plain NumPy array (with an attached metadata dictionary) - making it easy to write template code that can be used to plot different datasets without the need to manually modify titles and labels for each new piece of data.

In [ ]:
import earthkit.data as ekd
import earthkit.plots as ekp

### A quick example

Here is what the feature looks like in practice. A single format string is applied to a chart with two layers - temperature and pressure - and all placeholders resolve automatically:

In [ ]:
data = ekd.from_source("sample", "era5-2t-msl-1985122512.grib").to_fieldlist()
temperature, pressure = data

chart = ekp.Map(domain="North Atlantic")
chart.contourf(temperature, style="auto")
chart.contour(pressure, style="auto")
chart.legend(label="{variable_name} ({units})")
chart.coastlines()
chart.borders()
chart.gridlines()
# {time} resolves to the valid datetime; {variable_name} lists both fields
chart.title("ERA5 {variable_name} - {time:%H:%M UTC on %d %b %Y}")
chart.show()

Placeholders in these strigns can be anything from **GRIB keys** to **xarray Dataset attributes** or **scalar coordinate values**, or metadata keys from a dictionary passed at plot time. It can also be things like **units** or **domain**, which will update based on the arguments you pass to your plots.

#### "Magic" keys

In the example above, `{variable_name}` is not a direct attribute lookup. It searches several candidate attributes in priority order and returns the first one it finds:

```
long_name  →  standard_name  →  name  →  short_name
```

This makes `{variable_name}` work correctly across GRIB, netCDF, xarray, and NumPy without any extra configuration.

> NOTE: You can of course use GRIB keys or CF metadata keys directly, e.g. `standard_name` or `short_name`, depending on the metadata you have available. `variable_name` simply attempts to find the best key available tyo describe the variable name.

Similarly, `{time}` is also a magic key: it figures out the most appropriate time representation for the data automatically, so you don't need to know whether you have a forecast or observations or an analysis. Handling various components of weather forecast time metadata is covered later in this notebook.

### Deduplication - one template, multiple fields

When the same format string is applied across multiple layers or subplots, earthkit-plots collects the resolved value of each key from every field and applies a simple rule:

- **If all fields agree** on a value, it appears once.
- **If fields disagree**, the distinct values are joined into a list - e.g. `"2 metre temperature and Mean sea level pressure"`.

You already saw this in the example above: both layers share the same `{time}`, so it appears once in the title. But `{variable_name}` differs between temperature and pressure, so both names are listed.

## Time keys in a multi-panel forecast figure

The deduplication rules and time keys become especially useful in multi-panel forecast figures, where each panel shows a different time step but all panels share the same base time and variable.

The GRIB file below contains 9 time steps (every 6 hours over two days) of two variables: 10-metre wind gusts (`10fg6`) and mean sea-level pressure (`msl`). Let's look at the fields - notice the column names, which we can map directly to format keys:

In [ ]:
joachim = ekd.from_source(
    "url",
    "https://get.ecmwf.int/repository/test-data/metview/gallery/fc_msl_wg_joachim.grib",
).to_fieldlist()

joachim.ls()

The column names in the `ls()` output hint at the format keys available. Dotted names like `time.valid_datetime` can map directly to `{time.valid_datetime}` in a format string - no magic involved, just a direct lookup of that metadata field.

For time, the available keys are:

| Key | What it contains |
|-----|-----------------|
| `{time.valid_datetime}` | The explicit valid (forecast) datetime |
| `{time.valid_basetime}` | The model run time - when the forecast was initialised |
| `{time.step}` | Hours elapsed since the base time |

In the figure below, each panel receives fields at a different valid time, so `{time.valid_datetime}` and `{time.step}` vary per panel and each subplot title is different. `{time.valid_basetime}` is the same across all fields, so it appears just once in the figure title. `{parameter.name}` has two values across the 18 fields, so we will see both variables in the title:

In [ ]:
figure = ekp.Figure(domain=[-5, 23, 40, 58], figsize=(7, 7), rows=3, columns=3)

figure.contourf(joachim.sel({"parameter.variable": "10fg6"}), style="auto")
figure.contour(joachim.sel({"parameter.variable": "msl"}), units="hPa", style="auto")

figure.land()
figure.coastlines()
figure.borders()

figure.legend()
figure.subplot_titles("{time.valid_datetime:%Y-%m-%d %H} UTC (+{time.step}h)")
figure.title(
    "ECMWF HRES Run: {time.base_datetime:%Y-%m-%d %H} UTC\n{parameter.name}",
    y=1.1,
)

figure.show()

### Singleton coordinates - point data

When a coordinate has only a single value (a *scalar coordinate*), earthkit-plots makes it available as a format key named after that coordinate. This is most useful for single-point time series, where `latitude` and `longitude` are scalars.

The `%Lt` and `%Ln` specifiers format them with a degree symbol and N/S or E/W direction:

In [ ]:
era5_ts = ekd.from_source("sample", "era5-reading-timeseries.nc").to_xarray()

(
    ekp.timeseries
    .line(era5_ts, color="steelblue", units="celsius")
    .title("ERA5 hourly {variable_name}\n{latitude:%Lt} {longitude:%Ln}")
    .xticks(frequency="D", format="%d %B", period=True)
    .show()
)

### Overriding metadata at plot time

Pass `metadata=` to any plotting method to inject or override key/value pairs. This takes precedence over all file-level attributes, making it useful for NumPy arrays (which carry no metadata at all) or to add context not present in the file:

In [ ]:
from datetime import datetime

fl = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib").to_fieldlist()
lats, lons = fl.geography.latlons()
t2m = fl.to_numpy().squeeze()

meta = {
    "units": "K",
    "long_name": "2 metre temperature",
    "time": datetime(1994, 12, 1),
    "institution": "ECMWF",
}

chart = ekp.Map(domain="Europe")
chart.contourf(t2m, x=lons, y=lats, metadata=meta, units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} - {time:%B %Y}")
chart.attribution("Source: {institution}", location="lower right")
chart.show()

### Quick reference

| Key | What it resolves to |
|-----|---------------------|
| `{variable_name}` | Human-readable variable name - searches `long_name`, `standard_name`, `name`, `short_name` |
| `{parameter.name}` | Direct lookup of the `parameter.name` metadata field (GRIB) |
| `{units}` | Units after any conversion applied at plot time |
| `{time}` | **Magic key** - the time that best represents when the data is valid |
| `{time.valid_datetime:%d %B %Y}` | Explicit valid datetime, formatted with `strftime` codes |
| `{time.valid_basetime}` | Forecast base (run) time |
| `{time.step}` | Forecast lead time in hours |
| `{domain}` | Name of the map domain (named domains only) |
| `{crs}` | Name of the map projection |
| `{latitude:%Lt}` | Latitude with degree symbol and N/S direction |
| `{longitude:%Ln}` | Longitude with degree symbol and E/W direction |
| `{location:%c}` | City name (requires `reverse-geocode`) |
| `{location:%C}` | Country name (requires `reverse-geocode`) |
| `{ensemble_member}` | Ensemble member number |
| `{<any.key>}` | Any dotted metadata key present in the data |

### Exercise

The Joachim dataset was initialised at midnight on 15 December 2011. Using the figure above as a starting point:

1. Change the subplot titles to show the day of the week, the date with a named month and a year, the time and the lead time (for example, the first title should read something like "Thu 15 Dec 2011 00 UTC (+0h)")
2. Change the figure title so that it reads naturally, e.g. `Storm Joachim - ECMWF HRES initialised 15 December 2011 00 UTC`

You may find Python's [`strftime` format codes](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes) useful for formatting the dates.